<a href="https://colab.research.google.com/github/safar1-gg/mac237-labs/blob/main/lab1/MAC237_LAB1_S03_Authentication_And_Access_Control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAC 237 Graded Lab 1, Session 3: authentication, then access control

**Wednesday 23 September 2026. Room B124, or a browser, whichever you are sitting at.**

This notebook is the graded work for session 3 and the last graded work in Lab 1. Every cell
uses only the Python 3 standard library, so it runs unchanged in Google Colab in a browser or
in Python 3 on the Kali image in Room B124.

**Nothing in this notebook contacts any host outside it.** There are no accounts here but the
ones the notebook invents.

Three questions get asked in order on every request a system handles. Who do you say you are.
Can you prove it. And, only then, may you do this. Skip the middle one and the third is
theater. You write all three today, and then you measure the thing that decides which access
control model an organization can actually afford to run.

The rest of the hour in B124 is ungraded Kali practice, named in the lab document.

## Section 1. Your personalization token

Put your own student ID in `sid` and run the cell.

In [3]:
import hashlib, hmac, datetime

sid = '24271242'
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
TOKEN = hashlib.sha256((sid + '-' + stamp).encode()).hexdigest()[:16]
print('MAC237 Lab 1 token:', TOKEN)

MAC237 Lab 1 token: 77ca947691fa8717


## Section 2. Two models, two decision functions

Write the decision out as code rather than reading about it, because the difference between
these two models is not syntax. It is who is allowed to change the answer.

**Discretionary access control.** The owner of the file decides, by editing a list attached to
the file. That is exactly right for two people and exactly wrong for two thousand, for a
reason you will count in Section 5.

In [4]:
# owner -> file -> what everybody else may do, as r, w and - in fixed positions
ACL = {
    'alice': {'notes.txt': 'r--', 'draft.md': '---'},
    'bob':   {'sales.xlsx': 'rw-'},
    'erin':  {'audit.log': '---'},
}
BIT = {'read': 0, 'write': 1}

def dac(owner, subject, target, op):
    """The owner always holds full rights. Everybody else gets what the list says."""
    if subject == owner:
        return True
    perms = ACL.get(owner, {}).get(target, '---')
    return perms[BIT[op]] != '-'

print('files under discretionary control:')
for owner, files in sorted(ACL.items()):
    for name, perms in sorted(files.items()):
        print(f'  {name:12s} owner={owner:6s} others={perms}')

files under discretionary control:
  draft.md     owner=alice  others=---
  notes.txt    owner=alice  others=r--
  sales.xlsx   owner=bob    others=rw-
  audit.log    owner=erin   others=---


**Role-based access control.** Permissions attach to roles and people attach to roles, so what
an administrator edits when somebody changes job is one membership rather than every list that
person appears in. This is the model almost every organization above a certain size runs.

In [5]:
ROLE_PERMS = {
    'analyst': {'read': {'notes.txt', 'sales.xlsx'},              'write': set()},
    'auditor': {'read': {'notes.txt', 'sales.xlsx', 'audit.log'}, 'write': set()},
    'admin':   {'read': {'*'},                                    'write': {'*'}},
}
USER_ROLE = {'carol': 'analyst', 'dave': 'auditor', 'erin': 'admin'}

def rbac(subject, target, op):
    perms = ROLE_PERMS.get(USER_ROLE.get(subject, ''), {})
    allowed = perms.get(op, set())
    return target in allowed or '*' in allowed

for user, role in sorted(USER_ROLE.items()):
    reads = sorted(ROLE_PERMS[role]['read'])
    writes = sorted(ROLE_PERMS[role]['write'])
    print(f'  {user:6s} role={role:8s} reads={reads} writes={writes}')

  carol  role=analyst  reads=['notes.txt', 'sales.xlsx'] writes=[]
  dave   role=auditor  reads=['audit.log', 'notes.txt', 'sales.xlsx'] writes=[]
  erin   role=admin    reads=['*'] writes=['*']


## Section 3. The truth table

Six cases with the expected answer written down before the code ran. If a line reads FAIL,
your reading of the model and the function disagree, and the thing to fix is the function or
your reading. Do not edit the expected value to make the line go green.

In [6]:
CASES = [
    ('DAC  bob reads alice/notes.txt',      dac('alice', 'bob', 'notes.txt', 'read'),    True),
    ('DAC  bob writes alice/notes.txt',     dac('alice', 'bob', 'notes.txt', 'write'),   False),
    ('DAC  alice writes her own draft.md',  dac('alice', 'alice', 'draft.md', 'write'),  True),
    ('RBAC carol reads audit.log',          rbac('carol', 'audit.log', 'read'),          False),
    ('RBAC dave reads audit.log',           rbac('dave', 'audit.log', 'read'),           True),
    ('RBAC erin writes payroll.db',         rbac('erin', 'payroll.db', 'write'),         True),
]

print('token:', TOKEN)
print()
bad = 0
for name, got, want in CASES:
    ok = (got == want)
    bad += (not ok)
    print(f'{"PASS" if ok else "FAIL"}  {name:36s} decision={str(got):5s} expected={want}')

assert bad == 0, 'a decision function disagrees with the model it claims to implement'
print()
print('  CHECK PASS: all six decisions match the model')

token: 77ca947691fa8717

PASS  DAC  bob reads alice/notes.txt       decision=True  expected=True
PASS  DAC  bob writes alice/notes.txt      decision=False expected=False
PASS  DAC  alice writes her own draft.md   decision=True  expected=True
PASS  RBAC carol reads audit.log           decision=False expected=False
PASS  RBAC dave reads audit.log            decision=True  expected=True
PASS  RBAC erin writes payroll.db          decision=True  expected=True

  CHECK PASS: all six decisions match the model


## Section 4. Authentication has to come first

`rbac` answers may you. It never asked who you are, and it has no way to find out. On its own
it will happily authorize anybody who types a username that exists.

Below, the three steps are separated so you can watch each one refuse. Identification is the
claim. Authentication is the proof, here a message authentication code over a challenge,
computed with a key only that user and the system hold, which is the same construction you
built in session 2. Authorization is the decision from Section 3, and it is reached only if
the first two succeeded.

In [7]:
CREDENTIALS = {u: hashlib.sha256((TOKEN + '-' + u).encode()).digest() for u in USER_ROLE}

def identify(claim):
    """Who do you say you are. Establishes nothing on its own."""
    return claim if claim in CREDENTIALS else None

def authenticate(username, challenge, proof):
    """Prove it, by tagging the challenge with the key only you and the system hold."""
    if username is None:
        return False
    expected = hmac.new(CREDENTIALS[username], challenge, hashlib.sha256).digest()
    return hmac.compare_digest(expected, proof)

def request(claim, challenge, proof, target, op):
    user = identify(claim)
    if user is None:
        return 'refused: no such identity'
    if not authenticate(user, challenge, proof):
        return 'refused: identity not proved'
    return 'allowed' if rbac(user, target, op) else 'refused: not permitted by role'

challenge   = b'session-' + TOKEN.encode()
dave_proof  = hmac.new(CREDENTIALS['dave'],  challenge, hashlib.sha256).digest()
carol_proof = hmac.new(CREDENTIALS['carol'], challenge, hashlib.sha256).digest()

TRIALS = [
    ('dave proves he is dave, reads audit.log',
     request('dave', challenge, dave_proof, 'audit.log', 'read'), 'allowed'),
    ('carol proves she is carol, reads audit.log',
     request('carol', challenge, carol_proof, 'audit.log', 'read'),
     'refused: not permitted by role'),
    ('carol claims to be dave, offering her own proof',
     request('dave', challenge, carol_proof, 'audit.log', 'read'),
     'refused: identity not proved'),
    ('a name nobody holds, with a made up proof',
     request('mallory', challenge, b'\x00' * 32, 'audit.log', 'read'),
     'refused: no such identity'),
]

print('token:', TOKEN)
print()
for name, got, want in TRIALS:
    print(f'{"PASS" if got == want else "FAIL"}  {name:48s} {got}')
    assert got == want, name + ': got ' + got

print()
print('  CHECK PASS: each of the three steps refused exactly the request it is there to refuse')

token: 77ca947691fa8717

PASS  dave proves he is dave, reads audit.log          allowed
PASS  carol proves she is carol, reads audit.log       refused: not permitted by role
PASS  carol claims to be dave, offering her own proof  refused: identity not proved
PASS  a name nobody holds, with a made up proof        refused: no such identity

  CHECK PASS: each of the three steps refused exactly the request it is there to refuse


## Section 5. What it costs to change one person's job

The usual argument for role-based control is that it scales, which is true and is not an
explanation. The explanation is a count of edits, and the count is what this section produces.

Two ordinary events. A new auditor joins and needs everything auditors read. And a new file
appears that every auditor has to be able to read. Count the administrative actions each model
needs for each event, as the number of auditors grows.

In [8]:
AUDITOR_FILES = sorted(ROLE_PERMS['auditor']['read'])

def edits_new_auditor(model):
    """One person joins the audit team and needs everything auditors read."""
    return 1 if model == 'rbac' else len(AUDITOR_FILES)

def edits_new_file(model, auditors):
    """One new file appears that every auditor must be able to read."""
    return 1 if model == 'rbac' else auditors

print('token:', TOKEN)
print('files the auditor role reads:', ', '.join(AUDITOR_FILES))
print()
print(f'{"auditors":>9}  {"a new auditor joins":^27}  {"a new auditable file appears":^27}')
print(f'{"":>9}  {"DAC":>13} {"RBAC":>13}  {"DAC":>13} {"RBAC":>13}')
for n in (1, 5, 25, 100):
    print(f'{n:>9}  {edits_new_auditor("dac"):>13} {edits_new_auditor("rbac"):>13}  '
          f'{edits_new_file("dac", n):>13} {edits_new_file("rbac", n):>13}')

assert edits_new_auditor('rbac') == 1
assert edits_new_auditor('dac') == len(AUDITOR_FILES) == 3
assert edits_new_file('rbac', 100) == 1
assert edits_new_file('dac', 100) == 100
print()
print('  CHECK PASS: the role-based count stays at one while the discretionary count grows')
print('  token:', TOKEN)

token: 77ca947691fa8717
files the auditor role reads: audit.log, notes.txt, sales.xlsx

 auditors      a new auditor joins      a new auditable file appears
                     DAC          RBAC            DAC          RBAC
        1              3             1              1             1
        5              3             1              5             1
       25              3             1             25             1
      100              3             1            100             1

  CHECK PASS: the role-based count stays at one while the discretionary count grows
  token: 77ca947691fa8717


## Closing this session, and closing Lab 1

Three `CHECK PASS` lines must appear above.

At minute 45, whichever platform you are on:

1. **Save the notebook into your repository**, at
   `lab1/MAC237_LAB1_S03_Authentication_And_Access_Control.ipynb`. In Colab: File, then Save a
   copy in GitHub. On Kali: save it into your clone, then `git add -A`,
   `git commit -m "lab1 session 3 final"`, `git push`.
2. Open the commit on github.com, confirm that the `lab1` folder holds all three notebooks and
   all of your screenshots, and copy the commit URL. That URL is half of your submission.
3. Screenshots, both showing your own username or Google account name and your token:
   `lab1_s03_access_models.png`, the six PASS lines from Section 3 and the four from Section 4;
   and `lab1_s03_admin_cost.png`, the table from Section 5.
4. Sign out before you leave and capture `lab1_s03_logout.png`.

## What goes in the report

1. In Section 4, carol's request was refused for a different reason than the request that
   claimed to be dave. Name both reasons and say which one a well designed system should tell
   the user about and which one it should not, with a reason.
2. Copy the Section 5 table. Then state, in one sentence, the administrative cost that
   discretionary control imposes as an organization grows, using your own numbers.
3. Role-based control made both counts one. Name one thing it made harder that discretionary
   control made easy, and one situation in which you would still choose discretionary control.
4. Cite CTPE Chapter 11 on identity and access management.